# Engineering, Storage, and Inference Diagnostics

This atlas reports measurements that exist in the repository: queue state, checkpoint compatibility, archive status, disk pressure, and training-log coverage. It does not substitute simulated latent spaces, Hessians, or noise curves for unmeasured experiments.

## System state from authoritative artifacts


In [ ]:
#| label: engineering-state
import json
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("..")
queue_path = ROOT / "refine-logs/queue/queue_state.json"
audit_path = ROOT / "results/checkpoint_store/compatibility_audit.json"
catalog_path = ROOT / "results/checkpoint_store/catalog.jsonl"

state = json.loads(queue_path.read_text())
audit = json.loads(audit_path.read_text())
queue_counts = Counter(j.get("status", "unknown") for j in state["jobs"])
catalog_rows = [json.loads(line) for line in catalog_path.read_text().splitlines() if line.strip()]
catalog_counts = Counter(r.get("status", "unknown") for r in catalog_rows)

pd.DataFrame({
    "layer": ["queue"] * len(queue_counts) + ["checkpoint catalog"] * len(catalog_counts),
    "state": list(queue_counts) + list(catalog_counts),
    "count": list(queue_counts.values()) + list(catalog_counts.values())
})

Queue completion and archival compatibility answer different questions. The queue asks whether a process exited successfully. The compatibility audit asks whether the surviving model can support the current scientific comparison. The second is stricter and is the count used by the release gate.

## Content-pinned contract


In [ ]:
#| label: engineering-contract
contract = audit["contract"]
pd.DataFrame({
    "field": [
        "contract id", "source bundle SHA-256", "state schema SHA-256",
        "sample rate", "units", "normalization",
        "train records", "validation records", "test records"
    ],
    "value": [
        contract["contract_id"], contract["approved_source_bundle_sha256"],
        contract["state_schema_sha256"], contract["preprocessing"]["sample_rate_hz"],
        contract["preprocessing"]["units"], contract["preprocessing"]["normalization"],
        contract["split_content_roots"]["train"]["records"],
        contract["split_content_roots"]["val"]["records"],
        contract["split_content_roots"]["test"]["records"]
    ]
})

File names alone are insufficient because content can change without renaming a split. The contract therefore binds both inventory hashes and content-root hashes. The model state is stored at reduced precision only under a pinned state schema; inference must reconstruct the architecture from the sidecar and verify the digest before loading weights.

## Space-efficient checkpoint lifecycle

The checkpoint store implements a tiered lifecycle:

1. training writes a local checkpoint and provenance sidecar;
2. the archiver hashes both, uploads the immutable artifact, and verifies the remote copy;
3. the catalog records identity, digest, byte size, contract, and locator;
4. only after verification is the bulky local checkpoint evicted;
5. inference materializes the requested model by `model_id` into a bounded cache, verifies SHA-256, loads it, and then prunes the cache.

This preserves per-model inference while preventing hundreds of roughly checkpoint-sized files from filling the system disk. A database catalog is useful for lookup and transactional metadata, but storing all neural-network blobs inside one SQLite file would not reduce their information content and would make corruption or copying more expensive. The effective space saving comes from verified remote object storage, reduced-precision state, deduplication where hashes match, and a small materialization cache.

The inference CLI can enumerate every currently eligible identity with `--list-ready`. That list is derived from the compatibility audit rather than filenames. Each invocation then cross-checks the catalog digest against the audit, materializes at most one exact remote object, runs the model, and prunes the cache in a `finally` block. Consequently, a streaming pass over all eligible models requires space for one checkpoint plus any deliberately retained outputs, while a stale audit, wrong source bundle, incompatible model, or swapped digest stops before inference. The output tensor is optional: omitting `--output` performs the verified forward pass without writing a reconstruction file, so a smoke-test sweep can return both checkpoint-cache and output-retention costs to zero after every model. When an output is requested, it is flushed to a same-directory hidden temporary file and atomically renamed; an ordinary save failure therefore cannot expose a partial tensor under the final name. Input validation occurs before checkpoint retrieval. Retained reconstructions still need their own lifecycle policy; moving checkpoints out of the working tree does not make an unbounded output directory safe.


In [ ]:
#| label: checkpoint-storage-accounting
#| tbl-cap: 'Logical checkpoint volume versus bulky bytes currently retained on disk. Historical error rows are quarantined generations, not eligible inference models.'
eligible_ids = {
    model["model_id"] for model in audit["models"] if model.get("compatible")
}
eligible_catalog = [row for row in catalog_rows if row["model_id"] in eligible_ids]
assert len(eligible_catalog) == audit["counts"]["compatible"]
assert all(row["status"] in {"remote_verified", "cached"} for row in eligible_catalog)

catalog_logical_bytes = sum(int(row["size_bytes"]) for row in catalog_rows)
catalog_local_bytes = sum(
    int(row["size_bytes"]) for row in catalog_rows if row.get("local_path")
)
eligible_logical_bytes = sum(int(row["size_bytes"]) for row in eligible_catalog)
historical_rows = [row for row in catalog_rows if row["model_id"] not in eligible_ids]
median_eligible_checkpoint_bytes = int(pd.Series(
    [int(row["size_bytes"]) for row in eligible_catalog]
).median())

display(pd.DataFrame([
    ("All catalogued generations", len(catalog_rows), catalog_logical_bytes),
    ("Current inference-eligible generation", len(eligible_catalog), eligible_logical_bytes),
    ("Historical/quarantined generation", len(historical_rows), sum(int(row["size_bytes"]) for row in historical_rows)),
    ("Bulky checkpoint bytes currently local", None, catalog_local_bytes),
    ("Projected 480-model all-local store", 480, median_eligible_checkpoint_bytes * 480),
    ("One-checkpoint streaming-cache bound", 1, median_eligible_checkpoint_bytes),
], columns=["Storage layer", "Model identities", "Bytes"]))

The logical-byte total is how much exact model content the catalog can address; it is not current disk usage. The 480-model row is an explicit projection using the observed median size of a current eligible checkpoint, whereas the other rows are directly observed. An `error` row records a quarantined historical generation and is deliberately non-materializable. Current inference eligibility is the intersection of the compatibility audit and a `remote_verified` or `cached` catalog identity. This prevents a large historical row count from being mistaken for current checkpoint failures.

## All-compatible-model inference readiness


In [ ]:
#| label: compatible-inference-readiness
#| tbl-cap: 'Exact archived-model inference readiness on one real PTB-XL test record. Timings are operational observations, not model-quality endpoints.'
import hashlib

inference_root = ROOT / "results/factorial_mixed_level/inference_readiness"
inference_summary_path = inference_root / "summary.json"
inference_csv_path = inference_root / "per_model_inference_readiness.csv"
inference_lead_csv_path = inference_root / "per_model_per_lead_case_metrics.csv"
inference_code_path = ROOT / "scripts/benchmark_factorial_inference_readiness.py"
inference_summary = json.loads(inference_summary_path.read_text())
inference_csv_bytes = inference_csv_path.read_bytes()
inference_lead_csv_bytes = inference_lead_csv_path.read_bytes()

assert hashlib.sha256(inference_csv_bytes).hexdigest() == inference_summary["csv_sha256"]
assert hashlib.sha256(inference_lead_csv_bytes).hexdigest() == inference_summary["per_lead_case_metrics_csv_sha256"]
assert hashlib.sha256(audit_path.read_bytes()).hexdigest() == inference_summary["compatibility_audit_sha256"]
assert hashlib.sha256(inference_code_path.read_bytes()).hexdigest() == inference_summary["benchmark_code_sha256"]

inference_rows = pd.read_csv(inference_csv_path)
inference_lead_rows = pd.read_csv(inference_lead_csv_path)
assert len(inference_rows) == audit["counts"]["compatible"]
assert inference_summary["models_completed"] == len(inference_rows)
assert inference_rows.finite.all() and inference_summary["all_finite"]
assert inference_summary["cache_retained_bytes"] == 0
assert len(inference_lead_rows) == len(inference_rows) * 9
assert inference_lead_rows[["mse", "mae", "pearson", "variance_ratio"]].notna().all().all()

display(pd.DataFrame([
    ("Compatible identities expected", inference_summary["models_expected"]),
    ("Models strictly loaded and executed", inference_summary["models_completed"]),
    ("Finite reconstructions", int(inference_rows.finite.sum())),
    ("Input tensor SHA-256", inference_summary["input_sha256"]),
    ("Prepared input shape", str(inference_summary["prepared_input_shape"])),
    ("Reconstruction shape", str(inference_summary["output_shape"])),
    ("Logical checkpoint bytes traversed", inference_summary["checkpoint_logical_bytes"]),
    ("Checkpoint-cache bytes retained", inference_summary["cache_retained_bytes"]),
], columns=["Readiness gate", "Observed value"]))

display(inference_rows[[
    "model_id", "checkpoint_sha256", "load_and_materialize_seconds",
    "forward_median_seconds", "repeats", "output_mean", "output_std", "finite",
]])

Every identity compatible at the bound audit timestamp was reconstructed successfully from the actual PTB-XL test tensor `100.pt`. The gate table reports the cohort size, logical checkpoint bytes traversed, output shape, and zero retained cache bytes; the per-model table reports the observed load and CPU-forward timings. These measurements demonstrate operability under a one-thread CPU configuration. They are not throughput estimates, hardware-normalized comparisons, or clinical-quality measurements, and isolated timing excursions should not be interpreted as architectural effects from three repeats under concurrent system load.


In [ ]:
#| label: compatible-inference-index-case-lead-summary
#| tbl-cap: Missing-lead reconstruction diagnostics across the current compatible models for PTB-XL test record 100 only.
lead_order = ["III", "aVR", "aVL", "aVF", "V1", "V3", "V4", "V5", "V6"]
index_case_lead_summary = (
    inference_lead_rows.groupby("lead", as_index=False)
    .agg(
        models=("model_id", "nunique"),
        median_mse=("mse", "median"),
        minimum_mse=("mse", "min"),
        maximum_mse=("mse", "max"),
        median_pearson=("pearson", "median"),
        minimum_pearson=("pearson", "min"),
        maximum_pearson=("pearson", "max"),
        median_variance_ratio=("variance_ratio", "median"),
    )
    .set_index("lead")
    .loc[lead_order]
    .reset_index()
)
display(index_case_lead_summary)

In [ ]:
#| label: compatible-inference-index-case-heatmap
#| fig-cap: 'Per-model, per-missing-lead MSE on one indexed PTB-XL test record. This heatmap diagnoses output behavior and must not be read as cohort-level model ranking.'
import plotly.express as px

case_mse = (
    inference_lead_rows.pivot(index="model_id", columns="lead", values="mse")
    .reindex(columns=lead_order)
    .sort_index()
)
case_heatmap = px.imshow(
    case_mse,
    aspect="auto",
    color_continuous_scale="Viridis",
    labels={"x": "Reconstructed missing lead", "y": "Exact model identity", "color": "MSE"},
)
case_heatmap.update_layout(height=520)
case_heatmap.show()

The indexed case exposes why a single aggregate score is insufficient. In this rendered snapshot, V1 has the largest median MSE and a negative median correlation, while V3 and V4 retain high median correlations but show median variance ratios well above one. A waveform can therefore track timing while miscalibrating amplitude dispersion. This is genuine record-level evidence, but it is still one case selected for a systems readiness audit; it neither estimates population performance nor establishes a loss-component effect.

## Compatibility—not existence—is the inference gate


In [ ]:
#| label: compatible-model-table
model_rows = []
for m in audit["models"]:
    model_rows.append({
        "model_id": m.get("model_id"),
        "compatible": bool(m.get("compatible")),
        "reason": "; ".join(m.get("reasons", [])) or "approved",
        "checkpoint_sha256": str(m.get("checkpoint_sha256", ""))[:16]
    })
pd.DataFrame(model_rows).sort_values(
    ["compatible", "model_id"], ascending=[False, True]
).head(25)

An inference command should refuse a model if its digest, state schema, architecture descriptor, or preprocessing contract differs from the approved sidecar. That refusal is a safety feature: loading “a checkpoint with the same mask” is not equivalent to loading the model that was analyzed.

## Training-log diagnostics


In [ ]:
#| label: log-coverage
import re

log_dir = ROOT / "refine-logs/queue/logs"
logs = sorted(log_dir.glob("f_*_s*.log"))
rows = []
for path in logs:
    content = path.read_text(errors="replace")
    epochs = [int(x) for x in re.findall(r"Epoch\s+(\d+)", content)]
    rows.append({
        "model_id": path.stem,
        "bytes": path.stat().st_size,
        "last_epoch_seen": max(epochs) if epochs else None,
        "cuda_oom_mentions": content.lower().count("out of memory"),
        "traceback_mentions": content.count("Traceback")
    })
log_df = pd.DataFrame(rows)
pd.DataFrame({
    "quantity": ["log files", "logs with epoch markers", "OOM mentions", "tracebacks"],
    "value": [len(log_df), log_df.last_epoch_seen.notna().sum() if len(log_df) else 0,
              int(log_df.cuda_oom_mentions.sum()) if len(log_df) else 0,
              int(log_df.traceback_mentions.sum()) if len(log_df) else 0]
})

These counts diagnose execution, not model quality. OOM and traceback mentions must be linked back to job attempts; they must not be averaged into loss-effect estimates.

## Current-generation convergence curves

The log counter above includes historical attempts. The next analysis is stricter: `build_factorial_training_diagnostics.py` joins only models that pass the current compatibility audit, hashes each log, and binds every epoch row to its checkpoint and source-bundle digests.


In [ ]:
#| label: compatible-training-summary
#| tbl-cap: Optimization diagnostics for current-contract compatible checkpoints.
diagnostic_root = ROOT / "results/factorial_mixed_level/training_diagnostics"
epoch_curves = pd.read_csv(diagnostic_root / "compatible_epoch_curves.csv")
model_diagnostics = pd.read_csv(
    diagnostic_root / "compatible_model_summary.csv"
)
training_diagnostic_status = json.loads(
    (diagnostic_root / "summary.json").read_text()
)
operational_eta = pd.read_csv(
    diagnostic_root / "operational_eta_by_kernel.csv"
)
optimization_scale = pd.read_csv(
    diagnostic_root / "optimization_scale_by_energy_distance.csv"
)
controlled_kernel_training = pd.read_csv(
    diagnostic_root / "controlled_kernel_training_contrasts.csv"
)
training_builder_path = ROOT / "scripts/build_factorial_training_diagnostics.py"
assert hashlib.sha256(training_builder_path.read_bytes()).hexdigest() == (
    training_diagnostic_status["builder_code_sha256"]
)
for filename in (
    "compatible_epoch_curves.csv",
    "compatible_model_summary.csv",
    "optimization_scale_by_energy_distance.csv",
    "controlled_kernel_training_contrasts.csv",
    "operational_eta_by_kernel.csv",
):
    assert hashlib.sha256((diagnostic_root / filename).read_bytes()).hexdigest() == (
        training_diagnostic_status["files"][filename]
    )
assert training_diagnostic_status["compatible_models"] == audit["counts"]["compatible"]

model_diagnostics.assign(
    duration_minutes=model_diagnostics.duration_seconds / 60,
    val_mse_reduction_percent=-100 * model_diagnostics.val_mse_relative_change,
)[[
    "model_id", "mmd_kernel", "epochs", "duration_minutes",
    "first_val_mse", "last_val_mse", "val_mse_reduction_percent",
    "cuda_oom_mentions", "traceback_mentions", "nonfinite_mentions",
]]

### Operational completion estimate

The old 3.8-day estimate assumed 13.5 minutes for every model and counted incompatible historical completions. The live estimator below uses only compatible completed runs, estimates a separate median duration for each MMD-kernel level, weights those medians by the actual pending/running kernel mix, and subtracts elapsed time from the active job. It is an operational planning estimate, not a scientific result or confidence interval.


In [ ]:
#| label: compatible-operational-eta
#| tbl-cap: Single-GPU operational ETA from current-contract run durations and the live remaining kernel mix.
eta = training_diagnostic_status["operational_eta"]
display(pd.DataFrame([
    ("Compatible duration samples", training_diagnostic_status["compatible_models"]),
    ("Pending or running jobs", eta["remaining_pending_or_running_jobs"]),
    ("Estimated remaining hours", eta["estimated_remaining_hours"]),
    ("Estimated remaining days", eta["estimated_remaining_days"]),
    ("Estimated completion (UTC)", eta["estimated_completion_utc"]),
    ("Observed-min scenario (hours)", eta["observed_min_scenario_hours"]),
    ("Observed-max scenario (hours)", eta["observed_max_scenario_hours"]),
], columns=["ETA field", "Value"]))

display(operational_eta[[
    "mmd_kernel", "compatible_duration_samples", "observed_min_minutes",
    "observed_median_minutes", "observed_max_minutes", "pending_jobs",
    "running_jobs", "running_elapsed_minutes", "estimated_remaining_minutes",
]])

The observed-min/maximum scenarios propagate the fastest/slowest compatible duration seen at each kernel level; they are not uncertainty bounds. Kernel samples remain sparse, other active loss bits can affect runtime, and interruptions or hardware changes invalidate the calendar date. The estimate must therefore be regenerated as each compatible model arrives.

The compatible rows displayed above completed ten logged epochs without OOM, traceback, or non-finite-value markers. Their first-to-last validation-MSE changes are computed in the table rather than frozen into prose, so newly archived models do not make the chapter stale. A falling scheduled-horizon trajectory is evidence of numerical optimization progress, not evidence that epoch 10 is globally optimal or that one loss mask is clinically superior.


In [ ]:
#| label: compatible-validation-mse-curves
#| fig-cap: Validation MSE trajectories from current-contract compatible runs. Total composite loss is not plotted because its scale changes with active loss terms.
import plotly.express as px

fig = px.line(
    epoch_curves,
    x="epoch",
    y="val_mse",
    color="model_id",
    markers=True,
    labels={"val_mse": "Validation MSE", "epoch": "Epoch", "model_id": "Model ID"},
)
fig.update_layout(height=480, legend_title_text="Exact model identity")
fig.show()

In [ ]:
#| label: compatible-composite-scale-audit
#| tbl-cap: 'Observed composite-loss scale by energy-distance activation among current compatible runs. Groups are configuration-confounded diagnostics, not causal effects.'
display(optimization_scale)

The component-scale distinction matters. The mask decoder is bound to the implementation: position 5 activates empirical energy distance, whose contribution is divided by the 0.05 normalizer. The live table above recomputes ED-active and ED-off counts and scale summaries from every currently compatible log, instead of freezing an early-cohort count in prose. The controlled completed examples without derivative or VCG terms show that energy-distance activation—not derivative loss alone—can inflate the composite objective scale while validation MSE remains comparable. Total composite loss is meaningful only as the optimized objective within a fixed mask; comparing it across masks would mistake units and weighting for predictive quality. The common validation-MSE component is suitable for numerical convergence monitoring, while final loss-effect claims still require locked test endpoints and all seeds.

## Generation-bound temporal morphology gate

Target fiducials are extracted once from the immutable 2,198-record PTB-XL test content root and stored in a hash-bound Parquet cache. Every event carries its detector sample index as well as its feature value. For each compatible model, the watcher materializes the exact archived checkpoint, reconstructs V3 and V6 on CPU, extracts P/Q/R/S/T amplitudes and QT intervals, records detector failures and pairing coverage, writes a per-model CSV/JSON pair atomically, and prunes the checkpoint cache. The target cache avoids repeating identical delineation work for every model.

The sample-index schema exposed a previously hidden QT construction error: 6,707 duplicate QT rows reused the same detected QRS onset, with some later T offsets producing multi-second pseudo-QT intervals. Enforcing unique onset use removed the duplicates but a full-tail audit still found 105 values above 1,000 ms and values as short as 16 ms. The final structural rule requires a QRS onset, an R peak after that onset, and then the first T offset after the R peak but before the next detected R peak or QRS onset. This rejects pre-R and cross-beat offsets without clipping values merely because they look implausible. Both the helper and caller source bytes enter the extractor digest, so each semantic correction forces a new target-cache generation instead of silently changing the meaning of an existing hash.


In [ ]:
#| label: temporal-target-detector-eda
#| tbl-cap: 'Target-side detector availability and event distributions on all 2,198 PTB-XL test records. This is measured before any reconstruction is evaluated.'
target_eda_root = ROOT / "results/factorial_v4/temporal_target_detector_eda"
target_eda_summary_path = target_eda_root / "summary.json"
target_eda_csv_path = target_eda_root / "target_detector_feature_eda.csv"
target_subgroup_csv_path = target_eda_root / "target_detector_subgroup_coverage.csv"
target_cache_metadata_path = (
    ROOT / "results/factorial_v4/temporal_mmd_generation_bound"
    / "_target_ptb_xl_features.json"
)
if all(path.is_file() for path in (
    target_eda_summary_path, target_eda_csv_path, target_subgroup_csv_path
)):
    target_eda_summary = json.loads(target_eda_summary_path.read_text())
    target_cache_metadata = json.loads(target_cache_metadata_path.read_text())
    assert target_eda_summary["target_cache_parquet_sha256"] == target_cache_metadata["parquet_sha256"]
    assert hashlib.sha256(target_cache_metadata_path.read_bytes()).hexdigest() == target_eda_summary["target_cache_metadata_sha256"]
    assert hashlib.sha256((ROOT / "scripts/build_temporal_target_detector_eda.py").read_bytes()).hexdigest() == target_eda_summary["builder_code_sha256"]
    assert hashlib.sha256((ROOT / "data/ptb_xl/ptbxl_database.csv").read_bytes()).hexdigest() == target_eda_summary["ptbxl_metadata_sha256"]
    assert hashlib.sha256(target_eda_csv_path.read_bytes()).hexdigest() == target_eda_summary["csv_sha256"]
    assert hashlib.sha256(target_subgroup_csv_path.read_bytes()).hexdigest() == target_eda_summary["subgroup_csv_sha256"]
    target_detector_eda = pd.read_csv(target_eda_csv_path)
    target_detector_subgroups = pd.read_csv(target_subgroup_csv_path)
    assert len(target_detector_eda) == target_eda_summary["feature_rows"]
    assert len(target_detector_subgroups) == target_eda_summary["subgroup_rows"]
    display(target_detector_eda)
else:
    target_detector_eda = pd.DataFrame()
    target_detector_subgroups = pd.DataFrame()
    display(pd.DataFrame({"status": ["Corrected target-detector EDA is rebuilding"]}))

In [ ]:
#| label: temporal-target-detector-coverage
#| fig-cap: Target-side record detection coverage by lead and feature. Reconstruction pairing coverage cannot exceed or be interpreted independently of this baseline.
if target_detector_eda.empty:
    display(pd.DataFrame({"status": ["No corrected target-detector EDA artifact yet"]}))
else:
    target_coverage_plot = px.bar(
        target_detector_eda,
        x="clinical_feature",
        y="record_detection_coverage",
        color="lead",
        barmode="group",
        hover_data=[
            "events", "records_detected", "events_per_detected_record_median",
            "value_q01", "value_median", "value_q99",
            "inter_event_interval_ms_median",
        ],
        labels={
            "clinical_feature": "Target feature",
            "record_detection_coverage": "Records with at least one detected event",
        },
    )
    target_coverage_plot.update_yaxes(range=[0, 1])
    target_coverage_plot.update_layout(height=460)
    target_coverage_plot.show()

In [ ]:
#| label: temporal-target-detector-subgroups
#| tbl-cap: Target-detector coverage stratified by raw PTB-XL sex code and transparent age bins. Coverage is record-level; this is not a reconstruction-fairness result.
if target_detector_subgroups.empty:
    display(pd.DataFrame({"status": ["No corrected subgroup detector artifact yet"]}))
else:
    display(target_detector_subgroups.sort_values(
        ["subgroup_variable", "record_detection_coverage", "lead", "clinical_feature"]
    ))

This target-only ledger is the denominator diagnostic for every later reconstruction comparison. It separates absence of a target fiducial from reconstruction-side detector failure, and its value/timing tails expose extraction pathologies that record-level coverage alone would miss. Sex is deliberately reported as raw codes because the local table does not carry a trustworthy text mapping. Ages above 100 are isolated as invalid rather than merged into the oldest clinical group. These strata audit detector availability; they do not estimate model fairness.

`build_temporal_mmd_generation_summary.py` accepts a model artifact only when all of the following match: current evaluator bytes, compatible checkpoint digest, source bundle, training contract, test byte-content root, target-cache digest, emitted CSV digest, expected record denominator, row-level identity, finite metrics, and zero reconstruction-batch failures. The current evaluator uses monotonic one-to-one event matching that first maximizes match count and then minimizes total absolute detector-time error within a prespecified 100 ms tolerance. Each feature row must account exactly for paired and unmatched real/reconstructed events, and its 95th-percentile pairing error cannot exceed the tolerance. A model with an older evaluator hash remains visible in the exclusion ledger but cannot enter plots or factorial inference.

The distribution diagnostic is now explicit rather than implied by the evaluator's name. Within each lead-feature row, the evaluator sorts the finite paired target and reconstruction values, takes at most 512 evenly spaced quantile-index samples from each, chooses an RBF bandwidth as the median nonzero pairwise absolute distance in the combined sketch, and reports the biased squared RBF MMD. The bounded sketch avoids quadratic memory growth and contains no random sampling. MMD² is zero for identical sketches and increases as their empirical distributions separate, but its adaptive bandwidth means values should be compared within the same clinical feature—not treated as a common physical-unit scale across amplitudes and QT intervals.

Every accepted model must also carry a compact 26,376-row Parquet ledger: one row for each of 2,198 records × two evaluated leads × six features. It stores detector state, complete paired/unmatched counts, finite-pair sufficient statistics, record-level mean error and MAE, and timing-error summaries—not waveforms or checkpoints. The acceptance builder independently reconstructs the aggregate counts, means, and variances from this ledger. This makes patient-cluster summaries and reconstruction-side detector-failure strata possible while keeping diagnostic storage bounded.

Pairing sensitivity is evaluated at 25, 50, 75, 100, and 150 ms during the same detector pass. A compact 60-row table records coverage, paired/unmatched events, means, and variance ratios for two leads × six features × five tolerances. The gate requires paired counts to be monotone nondecreasing, unmatched counts to be monotone nonincreasing, and the 100 ms slice to reproduce the primary aggregate exactly. Thus tolerance analysis adds sufficient statistics rather than five copies of reconstructed ECGs.


In [ ]:
#| label: temporal-morphology-acceptance-gate
#| tbl-cap: 'Live acceptance state for generation-bound morphology artifacts. Partial accepted models are operational diagnostics, not a factorial comparison.'
temporal_root = ROOT / "results/factorial_v4/temporal_mmd_generation_bound"
temporal_status = json.loads((temporal_root / "accepted_summary.json").read_text())
temporal_models = pd.read_csv(temporal_root / "accepted_model_artifacts.csv")
temporal_features = pd.read_csv(temporal_root / "accepted_feature_summary.csv")
temporal_exclusions = pd.read_csv(temporal_root / "excluded_artifacts.csv")
temporal_pareto = pd.read_csv(
    temporal_root / temporal_status["outputs"]["controlled_kernel_pareto"]
)
temporal_dominance = pd.read_csv(
    temporal_root / temporal_status["outputs"]["controlled_kernel_dominance"]
)
temporal_builder_path = ROOT / "scripts/build_temporal_mmd_generation_summary.py"
assert hashlib.sha256(temporal_builder_path.read_bytes()).hexdigest() == (
    temporal_status["builder_code_sha256"]
)
for output_key, filename in temporal_status["outputs"].items():
    assert hashlib.sha256((temporal_root / filename).read_bytes()).hexdigest() == (
        temporal_status["output_sha256"][output_key]
    )
assert len(temporal_pareto) == temporal_status["controlled_kernel_pareto_rows"]
temporal_training_snapshot = temporal_root / temporal_status["outputs"][
    "training_diagnostic_snapshot"
]
assert hashlib.sha256(temporal_training_snapshot.read_bytes()).hexdigest() == (
    temporal_status["training_diagnostic_summary_sha256"]
)
temporal_training_status = json.loads(temporal_training_snapshot.read_text())
if temporal_status["pipeline_throughput"]["status"] == "observed_median_rate_estimate":
    assert temporal_training_status["compatible_models"] == (
        temporal_status["eligible_compatible_models"]
    )

pd.DataFrame([
    ("Current compatible checkpoints", temporal_status["eligible_compatible_models"]),
    ("Accepted evaluated checkpoints", temporal_status["accepted_models"]),
    ("Accepted feature rows", temporal_status["accepted_feature_rows"]),
    ("Excluded stale/invalid artifacts", temporal_status["excluded_artifacts"]),
    ("Complete for factorial inference", temporal_status["complete_for_factorial_inference"]),
    ("Evaluator SHA-256", temporal_status["evaluation_code_sha256"]),
    ("Target-cache SHA-256", temporal_status["target_feature_cache_sha256"]),
], columns=["Gate field", "Value"])

In [ ]:
#| label: temporal-evaluator-throughput
#| tbl-cap: 'Observed serial training-arrival and temporal-evaluator service rates. Catch-up is an operational projection under stable concurrent medians, not a scientific interval.'
throughput = temporal_status["pipeline_throughput"]
if throughput["status"] == "observed_median_rate_estimate":
    display(pd.DataFrame([
        ("Training median minutes/model", throughput["training_median_minutes_per_model"]),
        ("Evaluator median minutes/model", throughput["evaluation_median_minutes_per_model"]),
        ("Evaluator duration samples", throughput["evaluation_duration_samples"]),
        ("Training arrivals/hour", throughput["training_models_per_hour"]),
        ("Evaluator completions/hour", throughput["evaluation_models_per_hour"]),
        ("Net backlog drain/hour", throughput["net_backlog_drain_models_per_hour"]),
        ("Current compatible-minus-accepted backlog", throughput["backlog_models"]),
        ("Evaluator keeps pace at observed medians", throughput["evaluator_keeps_pace_at_observed_medians"]),
        ("Projected backlog catch-up hours", throughput["estimated_backlog_catchup_hours"]),
    ], columns=["Throughput field", "Observed value"]))
else:
    display(pd.DataFrame({"status": [throughput["status"]]}))

The live table above—not a frozen prose count—is the synchronized throughput snapshot. A positive net drain rate means the evaluator is currently faster than training and should eventually reach the live frontier under the stated serial, uninterrupted, stable-mix assumptions. A nonpositive rate means no finite catch-up time is justified. Either conclusion is operational rather than scientific: a heavier loss mix, CPU/GPU contention, evaluator-generation invalidation, or interruption can change both rates immediately.


In [ ]:
#| label: temporal-morphology-exclusions
#| tbl-cap: Artifacts excluded from the current evaluator generation.
if temporal_exclusions.empty:
    display(pd.DataFrame({"status": ["No exclusions"]}))
else:
    display(temporal_exclusions)

In [ ]:
#| label: temporal-morphology-coverage
#| fig-cap: 'Per-model detector/pairing coverage for accepted partial artifacts. Coverage is shown to diagnose measurement failure, not to rank loss masks before grid completion.'
if temporal_features.empty:
    display(pd.DataFrame({
        "status": ["No artifact yet passes the current evaluator and digest gate"]
    }))
else:
    import plotly.express as px
    coverage_plot = px.bar(
        temporal_features,
        x="clinical_feature",
        y="record_pair_coverage",
        color="model_id",
        facet_col="lead",
        barmode="group",
        hover_data=[
            "n_records_total", "n_records_real_detected",
            "n_records_recon_detected", "n_records_paired", "n_beats",
        ],
        labels={
            "clinical_feature": "Feature",
            "record_pair_coverage": "Records with a finite paired measurement",
        },
    )
    coverage_plot.update_yaxes(range=[0, 1])
    coverage_plot.update_layout(height=500, legend_title_text="Exact model identity")
    coverage_plot.show()

In [ ]:
#| label: temporal-morphology-partial-effect-diagnostics
#| fig-cap: 'Variance ratios for accepted partial artifacts. The horizontal line marks equal reconstructed/target variance; these are feature-detector diagnostics, not completed factorial effects.'
if temporal_features.empty:
    display(pd.DataFrame({"status": ["No accepted feature rows"]}))
else:
    variance_plot = px.bar(
        temporal_features,
        x="clinical_feature",
        y="variance_ratio",
        color="model_id",
        facet_col="lead",
        barmode="group",
        hover_data=[
            "mean_real", "mean_recon", "ba_robust_slope", "record_pair_coverage",
        ],
        labels={
            "clinical_feature": "Feature",
            "variance_ratio": "Detected reconstructed / target variance",
        },
    )
    variance_plot.add_hline(y=1.0, line_dash="dash", line_color="black")
    variance_plot.update_layout(height=500, legend_title_text="Exact model identity")
    variance_plot.show()

    distribution_plot = px.bar(
        temporal_features,
        x="clinical_feature",
        y="distribution_rbf_mmd2",
        color="model_id",
        facet_col="lead",
        barmode="group",
        hover_data=[
            "distribution_rbf_bandwidth", "distribution_mmd_real_samples",
            "distribution_mmd_recon_samples", "record_pair_coverage",
        ],
        labels={
            "clinical_feature": "Feature",
            "distribution_rbf_mmd2": "Biased squared RBF MMD",
        },
    )
    distribution_plot.update_layout(
        height=500, legend_title_text="Exact model identity"
    )
    distribution_plot.show()

    partial_diagnostic = pd.DataFrame([
        ("Accepted models", temporal_features.model_id.nunique()),
        ("Lead-feature rows", len(temporal_features)),
        ("Rows with variance ratio < 1", int((temporal_features.variance_ratio < 1).sum())),
        ("Variance-ratio range", f"{temporal_features.variance_ratio.min():.4f}–{temporal_features.variance_ratio.max():.4f}"),
        ("RBF MMD² range", f"{temporal_features.distribution_rbf_mmd2.min():.6f}–{temporal_features.distribution_rbf_mmd2.max():.6f}"),
        ("Pair-coverage range", f"{temporal_features.record_pair_coverage.min():.2%}–{temporal_features.record_pair_coverage.max():.2%}"),
    ], columns=["Partial diagnostic", "Observed value"])
    display(partial_diagnostic)

    coverage_failures = temporal_features.assign(
        records_without_pair=(
            temporal_features.n_records_total - temporal_features.n_records_paired
        )
    )[[
        "model_id", "lead", "clinical_feature", "n_records_total",
        "n_records_real_detected", "n_records_recon_detected",
        "n_records_paired", "records_without_pair", "record_pair_coverage",
    ]].sort_values("record_pair_coverage")
    display(coverage_failures)

    pairing_diagnostics = temporal_features.assign(
        unmatched_event_fraction=(
            temporal_features.n_unmatched_real_events
            + temporal_features.n_unmatched_recon_events
        ) / (
            temporal_features.n_real_beats_detected
            + temporal_features.n_recon_beats_detected
        ).clip(lower=1)
    )[[
        "model_id", "lead", "clinical_feature", "pairing_tolerance_ms",
        "median_abs_pairing_error_ms", "p95_abs_pairing_error_ms",
        "n_unmatched_real_events", "n_unmatched_recon_events",
        "unmatched_event_fraction",
    ]].sort_values("unmatched_event_fraction", ascending=False)
    display(pairing_diagnostics)

In [ ]:
#| label: temporal-morphology-per-record-ledger
#| tbl-cap: Per-record detector states and patient-clustered paired feature differences for the first accepted exact model. Intervals are descriptive percentile-bootstrap intervals.
if temporal_models.empty:
    display(pd.DataFrame({"status": ["No accepted per-record ledger"]}))
else:
    anchor_model_id = temporal_models.sort_values("model_id").iloc[0].model_id
    anchor_metadata = json.loads((temporal_root / f"{anchor_model_id}.json").read_text())
    anchor_per_record_path = temporal_root / anchor_metadata["per_record_parquet"]
    assert hashlib.sha256(anchor_per_record_path.read_bytes()).hexdigest() == (
        anchor_metadata["per_record_parquet_sha256"]
    )
    anchor_per_record = pd.read_parquet(anchor_per_record_path)
    assert len(anchor_per_record) == anchor_metadata["per_record_rows"] == 2198 * 2 * 6

    detector_strata = (
        anchor_per_record.groupby(
            ["lead", "clinical_feature", "detector_state"], as_index=False
        ).size()
    )
    detector_strata["record_fraction"] = detector_strata["size"] / 2198
    display(detector_strata.sort_values(
        ["lead", "clinical_feature", "size"], ascending=[True, True, False]
    ))

    ptb_patient_map = pd.read_csv(
        ROOT / "data/ptb_xl/ptbxl_database.csv", usecols=["ecg_id", "patient_id"]
    )
    paired_record = anchor_per_record.loc[
        anchor_per_record.n_finite_paired_events > 0
    ].copy()
    paired_record["ecg_id"] = paired_record.record_id.astype(int)
    paired_record = paired_record.merge(
        ptb_patient_map, on="ecg_id", how="left", validate="many_to_one"
    )
    assert paired_record.patient_id.notna().all()

    rng = np.random.default_rng(20260801)
    clustered_rows = []
    for (lead, feature), group in paired_record.groupby(["lead", "clinical_feature"]):
        patient_difference = group.groupby("patient_id").paired_mean_difference.mean()
        values = patient_difference.to_numpy(float)
        bootstrap = np.array([
            rng.choice(values, size=len(values), replace=True).mean()
            for _ in range(1000)
        ])
        clustered_rows.append({
            "model_id": anchor_model_id,
            "lead": lead,
            "clinical_feature": feature,
            "records_with_finite_pair": len(group),
            "patients_with_finite_pair": len(values),
            "equal_patient_mean_difference": values.mean(),
            "percentile_95_ci_low": np.percentile(bootstrap, 2.5),
            "percentile_95_ci_high": np.percentile(bootstrap, 97.5),
            "record_median_mae": group.paired_mae.median(),
            "record_p95_mae": group.paired_mae.quantile(0.95),
        })
    display(pd.DataFrame(clustered_rows))

The detector-state table keeps `target_only`, `reconstruction_only`, `both_detected`, and `neither_detected` records explicit instead of conditioning silently on successful pairs. The bootstrap first averages repeated ECGs within patient and then resamples patients with a fixed seed. These intervals quantify the accepted anchor's detector-conditioned mean feature difference; they are not multiplicity-adjusted, do not include records without a finite pair, and do not yet estimate factorial effects.


In [ ]:
#| label: temporal-morphology-tolerance-sensitivity
#| fig-cap: Pairing-coverage sensitivity across five prespecified temporal windows for the first accepted exact model.
if temporal_models.empty:
    display(pd.DataFrame({"status": ["No accepted tolerance-sensitivity artifact"]}))
else:
    sensitivity_path = temporal_root / anchor_metadata["tolerance_sensitivity_csv"]
    assert hashlib.sha256(sensitivity_path.read_bytes()).hexdigest() == (
        anchor_metadata["tolerance_sensitivity_csv_sha256"]
    )
    tolerance_sensitivity = pd.read_csv(sensitivity_path)
    assert len(tolerance_sensitivity) == 2 * 6 * 5
    sensitivity_plot = px.line(
        tolerance_sensitivity,
        x="pairing_tolerance_ms",
        y="record_pair_coverage",
        color="clinical_feature",
        facet_col="lead",
        markers=True,
        hover_data=[
            "n_paired_events", "n_unmatched_real_events",
            "n_unmatched_recon_events", "variance_ratio",
        ],
        labels={
            "pairing_tolerance_ms": "Pairing tolerance (ms)",
            "record_pair_coverage": "Records with a matched event",
            "clinical_feature": "Feature",
        },
    )
    sensitivity_plot.update_yaxes(range=[0, 1])
    sensitivity_plot.update_layout(height=520)
    sensitivity_plot.show()

    endpoints = tolerance_sensitivity[
        tolerance_sensitivity.pairing_tolerance_ms.isin([25.0, 150.0])
    ].pivot(
        index=["lead", "clinical_feature"],
        columns="pairing_tolerance_ms",
        values=["record_pair_coverage", "variance_ratio", "n_paired_events"],
    )
    endpoints.columns = [f"{metric}_{int(tolerance)}ms" for metric, tolerance in endpoints.columns]
    endpoints = endpoints.reset_index()
    endpoints["coverage_delta_150_minus_25ms"] = (
        endpoints.record_pair_coverage_150ms - endpoints.record_pair_coverage_25ms
    )
    endpoints["variance_ratio_delta_150_minus_25ms"] = (
        endpoints.variance_ratio_150ms - endpoints.variance_ratio_25ms
    )
    endpoints["paired_event_delta_150_minus_25ms"] = (
        endpoints.n_paired_events_150ms - endpoints.n_paired_events_25ms
    )
    display(endpoints)

The tolerance curves diagnose whether conclusions are driven by the event-association window. Wider windows must increase or preserve match counts by construction, but feature means and variance ratios can still move because different events enter the paired set. The sensitivity table intentionally does not recompute the quadratic RBF MMD at every window; its role is to expose pairing instability with compact sufficient statistics.

For the first accepted MSE-only baseline (`f_1000000_s42`), the diagnostics reject a simplistic universal-shrinkage story. Eleven of twelve detector-paired lead-feature rows have variance ratios below one, but V6 QT has a ratio of **1.653**. Its equal-patient mean QT difference is **−31.2 ms** with a descriptive 95% percentile interval of **−35.3 to −27.0 ms**, while V3 is **−12.6 ms** (**−14.5 to −10.7 ms**). Detector behavior is part of that result: QT is reconstruction-only in 82 V3 and 79 V6 records. Pairing sensitivity is substantial for some rows—V6 QT coverage rises from **79.44% at 25 ms to 92.31% at 150 ms**, and V6 S coverage from **85.58% to 99.04%**. These are precisely the cases where a single-window aggregate would overstate measurement stability. The V3 Q-amplitude row's squared RBF MMD is **0.521**, but adaptive feature-specific bandwidths prohibit ranking that value against QT or another feature as though they shared a common scale.


In [ ]:
#| label: temporal-morphology-model-snapshot
#| tbl-cap: 'Descriptive model-level morphology snapshot for accepted artifacts. Masks differ in multiple loss factors, so this table is not a causal contrast.'
if temporal_features.empty:
    display(pd.DataFrame({"status": ["No accepted model snapshots"]}))
else:
    model_snapshot = (
        temporal_features.groupby(["model_id", "model_mask", "mmd_kernel"], as_index=False)
        .agg(
            lead_feature_rows=("clinical_feature", "size"),
            minimum_pair_coverage=("record_pair_coverage", "min"),
            maximum_pair_coverage=("record_pair_coverage", "max"),
            median_variance_ratio=("variance_ratio", "median"),
            rows_below_unit_variance=("variance_ratio", lambda x: int((x < 1).sum())),
            maximum_variance_ratio=("variance_ratio", "max"),
            median_distribution_rbf_mmd2=("distribution_rbf_mmd2", "median"),
            maximum_distribution_rbf_mmd2=("distribution_rbf_mmd2", "max"),
        )
    )
    display(model_snapshot)

A variance ratio below one is consistent with reduced dispersion among the detector-paired values, but it does not identify the cause. Reconstruction smoothing, detector selection, unequal beat counts, order-based mispairing, or record exclusions can all contribute. A ratio above one is equally important counterevidence to any universal shrinkage narrative. At this partial stage the correct use is failure-mode discovery and evaluator validation, not selection of a winning loss mask.

The currently accepted generation is the only basis for these plots. Earlier order-truncated artifacts remain visible in the exclusion ledger but are not carried forward as evidence. During a code-generation transition, zero accepted rows is therefore the correct fail-closed result rather than an invitation to reuse otherwise plausible historical values.

### Controlled MMD-kernel family: optimization before morphology


In [ ]:
#| label: controlled-kernel-training-context
#| tbl-cap: Validation-MSE and runtime contrasts against kernel level 0 within the same seed and binary-factor prefix. Temporal acceptance reports whether the candidate can enter the morphology comparison below.
accepted_temporal_ids = set(temporal_models.model_id)
controlled_training_context = controlled_kernel_training.loc[
    controlled_kernel_training.baseline_model_id.isin(accepted_temporal_ids)
].copy()
controlled_training_context["baseline_temporal_accepted"] = (
    controlled_training_context.baseline_model_id.isin(accepted_temporal_ids)
)
controlled_training_context["candidate_temporal_accepted"] = (
    controlled_training_context.candidate_model_id.isin(accepted_temporal_ids)
)
display(controlled_training_context[[
    "seed", "binary_prefix", "baseline_model_id", "candidate_model_id",
    "candidate_kernel", "baseline_last_val_mse", "candidate_last_val_mse",
    "delta_last_val_mse", "baseline_duration_minutes",
    "candidate_duration_minutes", "delta_duration_minutes",
    "candidate_temporal_accepted",
]])

Each row above is a within-seed, within-binary-prefix comparison against that block's kernel-0 baseline. The table expands automatically as additional five-level blocks complete; it does not freeze the first block's validation deltas or runtimes in prose. Runtime, validation MSE, and morphology can order the same candidates differently, especially under concurrent system load. The temporal-acceptance column prevents a trained checkpoint from entering morphology claims before its full per-record evaluator artifacts pass the generation gate.


In [ ]:
#| label: controlled-kernel-feature-pareto
#| tbl-cap: 'Within-feature Pareto membership for every complete controlled kernel block. Metrics are absolute bias, distance from unit variance, feature-wise distribution MMD², and pair-coverage shortfall; no cross-feature scaling or weighted score is used.'
pareto_summary = (
    temporal_pareto.groupby(["model_id", "mmd_kernel"], as_index=False)
    .agg(
        lead_feature_rows=("clinical_feature", "size"),
        nondominated_rows=("pareto_nondominated", "sum"),
        rows_dominated=("pareto_nondominated", lambda x: int((~x).sum())),
        maximum_dominators=("dominated_by_count", "max"),
    )
)
display(pareto_summary)
display(temporal_dominance.sort_values(
    ["rows_dominated", "dominator_kernel", "dominated_kernel"],
    ascending=[False, True, True],
))

In [ ]:
#| label: controlled-kernel-feature-pareto-map
#| fig-cap: Feature-specific Pareto membership across the five kernel levels in each complete seed/binary-prefix block. A filled cell means that no other level in that same block is simultaneously no worse on all four metrics and strictly better on at least one for that lead-feature row.
pareto_plot_rows = temporal_pareto.assign(
    endpoint=temporal_pareto.lead + " · " + temporal_pareto.clinical_feature,
    block_label=(
        "s" + temporal_pareto.seed.astype(str)
        + " · b" + temporal_pareto.binary_prefix.astype(str).str.zfill(6)
    ),
    kernel_label="K" + temporal_pareto.mmd_kernel.astype(str),
)
pareto_plot_rows["block_kernel"] = (
    pareto_plot_rows.block_label + " · " + pareto_plot_rows.kernel_label
)
block_order = pareto_plot_rows[["seed", "binary_prefix", "block_label"]].drop_duplicates(
).sort_values(["seed", "binary_prefix"]).block_label.tolist()
row_order = [f"{block} · K{kernel}" for block in block_order for kernel in range(5)]
pareto_map = pareto_plot_rows.pivot(
    index="block_kernel", columns="endpoint", values="pareto_nondominated"
).reindex(index=row_order)
assert not pareto_map.isna().any().any()
pareto_figure = px.imshow(
    pareto_map.astype(int),
    aspect="auto",
    color_continuous_scale=[[0, "#ececec"], [1, "#2166ac"]],
    zmin=0,
    zmax=1,
    labels={"x": "Lead-feature endpoint", "y": "Controlled block · kernel", "color": "Nondominated"},
)
pareto_figure.update_layout(height=max(360, 28 * len(pareto_map)))
pareto_figure.show()

This Pareto definition is deliberately local to one seed, binary prefix, lead, and feature. The summary and dominance tables above recompute membership for every complete block rather than carrying forward counts from the first `100000*` block. A high nondominated count does not make a kernel a universal winner because the definition excludes validation MSE, runtime, common-pair MAE, uncertainty, and clinical importance. The output is a transparent partial order, not an effect estimate or rank aggregation.


In [ ]:
#| label: temporal-morphology-controlled-kernel-contrasts
#| tbl-cap: 'Available same-binary-prefix, same-seed kernel contrasts against MMD level 0. Every delta is candidate minus level 0 within the same lead and feature.'
kernel_input = temporal_features.copy()
kernel_input["mask_text"] = kernel_input.model_mask.astype(str).str.zfill(7)
kernel_input["binary_prefix"] = kernel_input.mask_text.str[:6]
kernel_input["mean_difference"] = kernel_input.mean_recon - kernel_input.mean_real
kernel_input["absolute_bias"] = kernel_input.mean_difference.abs()
kernel_input["abs_log_variance_deviation"] = np.log(
    kernel_input.variance_ratio
).abs()

controlled_rows = []
controlled_identities = []
for (seed, binary_prefix), group in kernel_input.groupby(["seed", "binary_prefix"]):
    identities = group[["model_id", "mmd_kernel"]].drop_duplicates()
    if 0 not in set(identities.mmd_kernel) or len(identities) < 2:
        continue
    baseline_id = identities.loc[identities.mmd_kernel.eq(0), "model_id"].iloc[0]
    baseline = group[group.model_id.eq(baseline_id)][[
        "lead", "clinical_feature", "absolute_bias",
        "abs_log_variance_deviation", "distribution_rbf_mmd2",
        "record_pair_coverage",
    ]]
    for identity in identities.loc[~identities.mmd_kernel.eq(0)].itertuples():
        candidate = group[group.model_id.eq(identity.model_id)][[
            "lead", "clinical_feature", "absolute_bias",
            "abs_log_variance_deviation", "distribution_rbf_mmd2",
            "record_pair_coverage",
        ]]
        merged = baseline.merge(
            candidate,
            on=["lead", "clinical_feature"],
            suffixes=("_kernel0", "_candidate"),
            validate="one_to_one",
        )
        assert len(merged) == 12
        merged.insert(0, "candidate_model_id", identity.model_id)
        merged.insert(0, "baseline_model_id", baseline_id)
        merged.insert(0, "candidate_kernel", int(identity.mmd_kernel))
        merged.insert(0, "binary_prefix", binary_prefix)
        merged.insert(0, "seed", int(seed))
        for metric in (
            "absolute_bias", "abs_log_variance_deviation",
            "distribution_rbf_mmd2", "record_pair_coverage",
        ):
            merged[f"delta_{metric}"] = (
                merged[f"{metric}_candidate"] - merged[f"{metric}_kernel0"]
            )
        controlled_rows.append(merged)
        controlled_identities.append((seed, binary_prefix, baseline_id, identity.model_id, int(identity.mmd_kernel)))

if controlled_rows:
    controlled_kernel_contrasts = pd.concat(controlled_rows, ignore_index=True)
    display(controlled_kernel_contrasts[[
        "seed", "binary_prefix", "baseline_model_id", "candidate_model_id",
        "candidate_kernel", "lead", "clinical_feature",
        "delta_absolute_bias", "delta_abs_log_variance_deviation",
        "delta_distribution_rbf_mmd2", "delta_record_pair_coverage",
    ]])
    contrast_direction_summary = (
        controlled_kernel_contrasts.groupby(
            ["candidate_model_id", "candidate_kernel"], as_index=False
        )
        .agg(
            lead_feature_rows=("clinical_feature", "size"),
            rows_lower_absolute_bias=("delta_absolute_bias", lambda x: int((x < 0).sum())),
            rows_closer_to_unit_variance=("delta_abs_log_variance_deviation", lambda x: int((x < 0).sum())),
            rows_lower_distribution_mmd2=("delta_distribution_rbf_mmd2", lambda x: int((x < 0).sum())),
            rows_higher_pair_coverage=("delta_record_pair_coverage", lambda x: int((x > 0).sum())),
            largest_pair_coverage_loss=("delta_record_pair_coverage", "min"),
        )
    )
    display(contrast_direction_summary)
else:
    controlled_kernel_contrasts = pd.DataFrame()
    display(pd.DataFrame({"status": ["No accepted same-prefix contrast against kernel 0"]}))

In [ ]:
#| label: temporal-morphology-controlled-kernel-patient-contrasts
#| tbl-cap: Equal-patient change in record-level paired-event MAE for accepted controlled kernel contrasts. Negative values favor the candidate kernel on the common finite-pair subset. Intervals are unadjusted descriptive percentile bootstraps.
def verified_per_record(model_id):
    metadata = json.loads((temporal_root / f"{model_id}.json").read_text())
    path = temporal_root / metadata["per_record_parquet"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == metadata[
        "per_record_parquet_sha256"
    ]
    frame = pd.read_parquet(path)
    assert len(frame) == metadata["per_record_rows"] == 2198 * 2 * 6
    return frame

patient_kernel_rows = []
if controlled_identities:
    patient_map = pd.read_csv(
        ROOT / "data/ptb_xl/ptbxl_database.csv", usecols=["ecg_id", "patient_id"]
    )
    ledger_cache = {}
    for _, binary_prefix, baseline_id, candidate_id, candidate_kernel in controlled_identities:
        for model_id in (baseline_id, candidate_id):
            if model_id not in ledger_cache:
                ledger_cache[model_id] = verified_per_record(model_id)
        baseline = ledger_cache[baseline_id][[
            "record_id", "lead", "clinical_feature", "paired_mae",
            "n_finite_paired_events",
        ]]
        candidate = ledger_cache[candidate_id][[
            "record_id", "lead", "clinical_feature", "paired_mae",
            "n_finite_paired_events",
        ]]
        common = baseline.merge(
            candidate,
            on=["record_id", "lead", "clinical_feature"],
            suffixes=("_kernel0", "_candidate"),
            validate="one_to_one",
        )
        common = common.loc[
            common.n_finite_paired_events_kernel0.gt(0)
            & common.n_finite_paired_events_candidate.gt(0)
            & np.isfinite(common.paired_mae_kernel0)
            & np.isfinite(common.paired_mae_candidate)
        ].copy()
        common["ecg_id"] = common.record_id.astype(int)
        common = common.merge(patient_map, on="ecg_id", validate="many_to_one")
        common["delta_paired_mae"] = (
            common.paired_mae_candidate - common.paired_mae_kernel0
        )
        for feature_index, ((lead, feature), group) in enumerate(
            common.groupby(["lead", "clinical_feature"])
        ):
            patient_values = group.groupby("patient_id").delta_paired_mae.mean().to_numpy(float)
            rng = np.random.default_rng(
                20260801 + candidate_kernel * 100 + feature_index
            )
            bootstrap = np.array([
                rng.choice(patient_values, size=len(patient_values), replace=True).mean()
                for _ in range(1000)
            ])
            coverage_delta = controlled_kernel_contrasts.loc[
                controlled_kernel_contrasts.candidate_model_id.eq(candidate_id)
                & controlled_kernel_contrasts.lead.eq(lead)
                & controlled_kernel_contrasts.clinical_feature.eq(feature),
                "delta_record_pair_coverage",
            ].iloc[0]
            patient_kernel_rows.append({
                "binary_prefix": binary_prefix,
                "baseline_model_id": baseline_id,
                "candidate_model_id": candidate_id,
                "candidate_kernel": candidate_kernel,
                "lead": lead,
                "clinical_feature": feature,
                "feature_unit": "ms" if feature == "QT_Interval_ms" else "mV",
                "common_records_with_finite_pair": len(group),
                "common_patients_with_finite_pair": len(patient_values),
                "equal_patient_mean_delta_paired_mae": patient_values.mean(),
                "percentile_95_ci_low": np.percentile(bootstrap, 2.5),
                "percentile_95_ci_high": np.percentile(bootstrap, 97.5),
                "full_denominator_pair_coverage_delta": coverage_delta,
            })
    display(pd.DataFrame(patient_kernel_rows))
else:
    display(pd.DataFrame({"status": ["No controlled patient-level kernel contrast available"]}))

The first controlled accepted family is now complete: `1000000`–`1000004`, all at seed 42 with identical binary factors. The levels are none, global adaptive RBF, anatomical Laplacian, anatomical multiscale IMQ, and temporal K-means multiscale IMQ. Direction counts are deliberately not collapsed into a score. Relative to level 0, levels 1–4 lower absolute aggregate bias in **6/12, 9/12, 8/12, and 8/12** rows; move variance closer to one in **9/12, 10/12, 10/12, and 11/12**; lower feature-wise distribution MMD² in **7/12, 10/12, 10/12, and 9/12**; and improve full-denominator pair coverage in **8/12, 3/12, 7/12, and 8/12** rows. No kernel dominates all four axes.

The patient-level contrasts make the trade-offs more concrete. Anatomical Laplacian produces the largest V6-QT common-pair MAE decrease, about **20.3 ms**, but loses **6.64 percentage points** of full-denominator pair coverage. Global RBF lowers the same MAE by about **15.8 ms** while increasing coverage by **1.23 points**, even though its aggregate mean QT bias changes sign and grows in absolute value. Anatomical multiscale IMQ lowers V6-QT MAE by **8.2 ms** with a **0.41-point** coverage loss, while temporal K-means multiscale IMQ lowers it by **14.3 ms** with a **1.36-point** loss. These common-pair MAE changes coexist with slightly worse final validation MSE for all four candidates. Aggregate bias, record-level MAE, distribution discrepancy, detector availability, and validation MSE therefore answer different questions; none can serve as a single winner metric.

These intervals are unadjusted, detector-conditioned, and limited to one seed and the records where both models yield finite pairs. They are useful for debugging the measurement contract and exposing endpoint discordance, not for declaring a kernel effect. Confirmatory inference still requires all seed blocks, prespecified contrasts, multiplicity control, and explicit handling of differential detector failure.

The current matcher resolves one major ambiguity but not every measurement problem. The full-denominator ledger, detector-state strata, patient-cluster summaries, and five-window sensitivity grid now expose several ways the 100 ms result can change. They do not prove that any window is clinically correct: the detector can move or omit fiducials systematically, no clinician-adjudicated fiducial subset is available, and the compact ledger cannot retrospectively evaluate an arbitrary unprespecified window. Confirmatory QT/amplitude conclusions still require completion across masks and seeds, multiplicity-aware patient-cluster inference, and detector validation against independently adjudicated fiducials.

## Diagnostics still missing

Latent embeddings, kernel Gram matrices, gradient conflicts, Hessian spectra, noise response surfaces, calibration curves, and inference latency are worthwhile only when generated from named checkpoints and named records. Each future diagnostic must include its cohort, sampling rule, checkpoint digest, evaluation-code digest, random seed where applicable, and uncertainty. Until those artifacts exist, the book records the protocol rather than drawing decorative synthetic plots.